## FRAMEWORK INPUT

This document is responsable for:
- performing basic data preprocessing in all different .xlsx files
- merging those files into a big tabular dataset with configuration : timestamp, processID, attribute_1 ... attribute_n, event

In [16]:
import pandas as pd
import os
import pickle

In [17]:
FILE_PATH = "Data/Tabelas/OneDrive_1_09-07-2024/"
DATASET_PATH = "Datasets/"

In [18]:
def read_file(file_path, delimiter=';', skiprows=0):
    """
    Reads a file (CSV, RPT, or XLSX) into a Pandas DataFrame based on its extension.
    :param file_path: Path to the file
    :param delimiter: Delimiter used for CSV and RPT files (default is ';')
    :param skiprows: Number of rows to skip (default is 0)
    :return: Pandas DataFrame
    """
    try:
        file_extension = os.path.splitext(file_path)[1].lower()
                
        if file_extension in ['.csv', '.rpt']:
            df = pd.read_csv(file_path, delimiter=delimiter, engine='python', skiprows=skiprows)
        elif file_extension == '.xlsx':
            df = pd.read_excel(file_path, engine='openpyxl')
        else:
            raise ValueError("Unsupported file format")
        
        print(f"File {file_path} loaded successfully!")
        return df
    except Exception as e:
        print(f"Error reading file {file_path}: {e}")
        return None

In [19]:
def describe_dataset(df):
    """
    Provides a general description of the dataset.
    :param df: Pandas DataFrame
    """
    if df is not None:
        print("Dataset Information:")
        print(df.info())
        print("\nSummary Statistics:")
        print(df.describe())
        print("\nMissing Values:")
        print(df.isnull().sum())
    else:
        print("No dataset to describe.")

In [20]:
############################
#       LOAD DATASET
############################
report = read_file(FILE_PATH + "TPressReport.xlsx", delimiter=',')
hard = read_file(FILE_PATH + "TPressHard.xlsx", delimiter=',')
setup = read_file(FILE_PATH + "TPressSetup.xlsx", delimiter=',')
thick = read_file(FILE_PATH + "TPressTick.xlsx", delimiter=',')
weight = read_file(FILE_PATH + "TPressWeight.xlsx", delimiter=',')
masked = read_file(FILE_PATH + "Process1580_1067_masked.xlsx", delimiter=',')
var = read_file(FILE_PATH + "Var.xlsx", delimiter=',')


event = read_file(FILE_PATH + "TPressEvent.rpt", delimiter=';', skiprows=13)
parameter = read_file(FILE_PATH + "TPressParameter.rpt", delimiter=';', skiprows=11)


File Data/Tabelas/OneDrive_1_09-07-2024/TPressReport.xlsx loaded successfully!
File Data/Tabelas/OneDrive_1_09-07-2024/TPressHard.xlsx loaded successfully!
File Data/Tabelas/OneDrive_1_09-07-2024/TPressSetup.xlsx loaded successfully!
File Data/Tabelas/OneDrive_1_09-07-2024/TPressTick.xlsx loaded successfully!
File Data/Tabelas/OneDrive_1_09-07-2024/TPressWeight.xlsx loaded successfully!
File Data/Tabelas/OneDrive_1_09-07-2024/Process1580_1067_masked.xlsx loaded successfully!
File Data/Tabelas/OneDrive_1_09-07-2024/Var.xlsx loaded successfully!
File Data/Tabelas/OneDrive_1_09-07-2024/TPressEvent.rpt loaded successfully!
File Data/Tabelas/OneDrive_1_09-07-2024/TPressParameter.rpt loaded successfully!


In [21]:
######################################################################################## 
#                  B A S I C  D A T A  P R E P R O C E S S I N G                       #
########################################################################################
print('# Basic Data Preprocessing\n')

# remove irrelevant columns
process = masked.drop(axis=1, columns=["ProcessLocal", "ProcessCode", "ProcessFase", "ProcessArea"])
tpress_setup = setup.drop(axis=1, columns=["TPressSetupAuto"])
tpress_parameter = parameter.drop(axis=1, columns=["TPressParameterAuto"])
tpress_hard = hard.drop(axis=1, columns=["TPressHardAuto", "TPressHardSampleID", "TPressHardQProd"]) 
tpress_thick = thick.drop(axis=1, columns=["TPressThickAuto", "TPressThickSampleID", "TPressThickQProd"])
tpress_weight = weight.drop(axis=1, columns=["TPressWeightAuto", "TPressWeightSampleID", "TPressWeightQProd"])
tpress_report = report.drop(axis=1, columns=["TPressProdReportAuto", "TPressProdReportSampleID"])
event = event.drop(axis=1, columns=["EventAuto", "EventSubCode"])
print("Irrelevant columns removed.")

# remove duplicated entries
process = process.drop_duplicates()
tpress_setup = tpress_setup.drop_duplicates()
tpress_parameter = tpress_parameter.drop_duplicates()
event = event.drop_duplicates()
tpress_hard = tpress_hard.drop_duplicates(subset=["TPressHardProcess", "TPressHardDateTime"])
tpress_thick = tpress_thick.drop_duplicates(subset=["TPressThickProcess", "TPressThickDatetime"])
tpress_weight = tpress_weight.drop_duplicates(subset=["TPressWeightProcess", "TPressWeightDateTime"])
tpress_report = tpress_report.drop_duplicates(subset=["TPressProdReportProcess", "TPressProdReportDateTime"])
print("Duplicated columns removed.")

# remover as colunas que tenham apenas um valor único
tpress_setup = tpress_setup.loc[:, tpress_setup.nunique() > 1]
tpress_parameter = tpress_parameter.loc[:, tpress_parameter.nunique() > 1]
print("Single-value columns removed.")

# ensure the dates are in the correct format
tpress_parameter["TPressParameterDateTime"] = pd.to_datetime(tpress_parameter['TPressParameterDateTime']).dt.strftime('%Y-%m-%d %H:%M:%S')
tpress_parameter["TPressParameterDateTime"] = pd.to_datetime(tpress_parameter["TPressParameterDateTime"])
event["EventDateTime"] = pd.to_datetime(event["EventDateTime"])
print("Datetime columns correctly formated.")

# remove the codes related with the diameter
diam_codes = [35, 150, 151, 152, 153, 154, 155, 156, 187]
tpress_setup = tpress_setup[~tpress_setup["TPressSetupCode"].isin(diam_codes)]
tpress_parameter = tpress_parameter[~tpress_parameter["TPressParameterCode"].isin(diam_codes)]
print("Diameter-related codes removed from columns.")

# tpress_parameter tem null values: uma linha, onde todos os atributos são null --> delete
tpress_parameter = tpress_parameter.dropna()

# corrigir o mau formato da coluna TPressParameterValue, da tabela tpress_parameter
def convert_value(value):
    value = value.replace(',', '.')  # Replace commas with periods

    float_value = float(value)  # Convert to float
    if float_value.is_integer():  # Check if it's a whole number
        return int(float_value)  # Return as integer
    return float_value  # Return as float

tpress_parameter['TPressParameterValue'] = tpress_parameter['TPressParameterValue'].apply(convert_value)
print("Parameter values formatted correctly for tpress_parameter table.") 

# remover entradas de final de ocorrência de alarme
event = event[event["EventBeginEnd"] == 1]
event = event.drop(axis=1, columns="EventBeginEnd").reset_index(drop=True)
print("Filtered begin timestamp for alarms.")   

# remover alarmes inúteis
USELESS_ALARMS = [10001, 10002]  # TODO acrescentar os alarmes que não se querem/querem-se prever
event = event[~event["EventCode"].isin(USELESS_ALARMS)] 
print("Filtered the relevant alarms for prediction.") 
print(100 * '-')    

# Basic Data Preprocessing

Irrelevant columns removed.
Duplicated columns removed.
Single-value columns removed.
Datetime columns correctly formated.
Diameter-related codes removed from columns.
Parameter values formatted correctly for tpress_parameter table.
Filtered begin timestamp for alarms.
Filtered the relevant alarms for prediction.
----------------------------------------------------------------------------------------------------


In [22]:
######################################################################
#                   M E R G I N G  T A B L E S                       #
######################################################################

#####################################
# Merge attributes: tpress_attributes
#####################################

# Sample column renaming, assuming the actual names vary
tpress_hard.rename(columns={'TPressHardProcess': 'ProcessId', 
                            'TPressHardDateTime': 'DateTime'}, inplace=True)
tpress_thick.rename(columns={'TPressThickProcess': 'ProcessId', 
                             'TPressThickDatetime': 'DateTime'}, inplace=True)
tpress_weight.rename(columns={'TPressWeightProcess': 'ProcessId', 
                              'TPressWeightDateTime': 'DateTime'}, inplace=True)

# Outer merge on 'Process' and 'SampleID'
tpress_attributes = tpress_hard.merge(tpress_thick, on=['ProcessId', 'DateTime'], how='inner')\
                       .merge(tpress_weight, on=['ProcessId', 'DateTime'], how='inner')

# Fill 'Object' column with non-missing values from 'TPressHardObject', 'TPressThickObject', and 'TPressWeightObject' (the values in these columns are consistent and not contradictory: checked!)
tpress_attributes["Object"] = tpress_attributes['TPressHardObject'].fillna(tpress_attributes['TPressThickObject']).fillna(tpress_attributes['TPressWeightObject'])
tpress_attributes = tpress_attributes.drop(axis=1, columns=["TPressHardObject", "TPressThickObject", "TPressWeightObject"])

# Sort the rows
tpress_attributes = tpress_attributes.sort_values(by=['ProcessId', 'DateTime']).reset_index(drop=True)

# save dataset
store_path = DATASET_PATH + "tpress_attributes.pkl"
tpress_attributes.to_pickle(store_path)

print("Merged tpress_hard, tpress_thick, and tpress_weight on ['ProcessId', 'DateTime'] -> tpress_attributes")

#######################################################
# Merge attributes and report: tpress_attributes_report
#######################################################
tpress_report.rename(columns={'TPressProdReportProcess': 'ProcessId', 
                              'TPressProdReportDateTime': 'DateTime',
                              'TPressProdReportQProd': 'QProd'}, inplace=True)

tpress_attributes_report = pd.merge(
    tpress_attributes,
    tpress_report,
    on=['ProcessId', 'DateTime'],
    how="left" 
)

# Fill 'Object' column with non-missing values from 'TPressProdReportObject'
tpress_attributes_report["Object"] = tpress_attributes_report['Object'].fillna(tpress_attributes_report['TPressProdReportObject'])
tpress_attributes_report = tpress_attributes_report.drop(axis=1, columns=["TPressProdReportObject"])

# Sort the rows
tpress_attributes_report = tpress_attributes_report.sort_values(by=['ProcessId', 'DateTime']).reset_index(drop=True)

# save dataset
store_path = DATASET_PATH + "tpress_attributes_report.pkl"
#tpress_attributes_report.to_pickle(store_path)

print("Merged tpress_attributes, and tpress_report on ['ProcessId', 'DateTime'] -> tpress_attributes_report")

#######################################################
# merging tpress_attributes_report with tpress_setup (question: follow aproach from main_bial??)
#######################################################

tpress_setup.rename(columns={'TPressSetupProcess': 'ProcessId', 
                             'TPressSetupDateTime': 'DateTime'}, inplace=True)
tpress_attributes_report_setup = pd.merge(
    tpress_attributes_report,
    tpress_setup, 
    on=['ProcessId', 'DateTime'], 
    how='left'
)
# save dataset
store_path = DATASET_PATH + "tpress_attributes_report_setup.pkl"
tpress_attributes_report_setup.to_pickle(store_path)

print("Merged tpress_attributes_report, and tpress_setup on ['ProcessId', 'DateTime'] -> tpress_attributes_report")



Merged tpress_hard, tpress_thick, and tpress_weight on ['ProcessId', 'DateTime'] -> tpress_attributes
Merged tpress_attributes, and tpress_report on ['ProcessId', 'DateTime'] -> tpress_attributes_report
Merged tpress_attributes_report, and tpress_setup on ['ProcessId', 'DateTime'] -> tpress_attributes_report


In [23]:
# see the last generated table
tpress_attributes_report_setup.head()

,ProcessId,DateTime,TPressHardSample1,TPressHardSample2,TPressHardSample3,TPressHardSample4,TPressHardSample5,TPressHardSample6,TPressHardSample7,TPressHardSample8,...,TPressProdReportThickVM,TPressProdReportThickSrel,TPressProdReportHardVM,TPressProdReportHardSrel,TPressProdReportDiamVM,TPressProdReportDiamSrel,QProd,TPressSetupObject,TPressSetupCode,TPressSetupValue
0,207237,2022-01-05 21:27:48,75.0,88.0,79.0,47.0,81.0,89.0,87.0,80.0,...,4.48,0.55,79.4,15.77,0.0,0.0,0.0,1580.0,14.0,0.0
1,207237,2022-01-05 21:27:48,75.0,88.0,79.0,47.0,81.0,89.0,87.0,80.0,...,4.48,0.55,79.4,15.77,0.0,0.0,0.0,1580.0,16.0,0.0
2,207237,2022-01-05 21:27:48,75.0,88.0,79.0,47.0,81.0,89.0,87.0,80.0,...,4.48,0.55,79.4,15.77,0.0,0.0,0.0,1580.0,17.0,0.0
3,207237,2022-01-05 21:27:48,75.0,88.0,79.0,47.0,81.0,89.0,87.0,80.0,...,4.48,0.55,79.4,15.77,0.0,0.0,0.0,1580.0,18.0,1.8
4,207237,2022-01-05 21:27:48,75.0,88.0,79.0,47.0,81.0,89.0,87.0,80.0,...,4.48,0.55,79.4,15.77,0.0,0.0,0.0,1580.0,19.0,2.3


In [24]:
#######################################
# Anomaly Definition - merging events #
#######################################

# Paradigm:
# See from each timestamp on last table (tpress_attributes_report in this version)
#           if a given event (see event table) occurs after n minutes : event = 1, otherwise event = 0 :==> event will be the variable that we are trying to predict
# 

WIN_SIZE = 30 # number in minutes
last_dataset = tpress_attributes_report_setup # change this when you merge other tables before

# converting datetime of last_dataset to a datetime
last_dataset["DateTime"] = pd.to_datetime(last_dataset["DateTime"])

# merge 
db_with_events = last_dataset.merge(event, left_on="ProcessId", right_on="EventProcess")


db_with_events["event"] = ((db_with_events["EventDateTime"] >= db_with_events["DateTime"]) & 
                   (db_with_events["EventDateTime"] <= db_with_events["DateTime"] + pd.Timedelta(minutes=WIN_SIZE))).astype(int)

# Keep only the relevant columns
result = db_with_events.drop(columns=["EventProcess", "EventDateTime"]).drop_duplicates()


# save dataset
store_path = DATASET_PATH + "final_dataset.pkl"
#result.to_pickle(store_path)

In [25]:
result.head()

,ProcessId,DateTime,TPressHardSample1,TPressHardSample2,TPressHardSample3,TPressHardSample4,TPressHardSample5,TPressHardSample6,TPressHardSample7,TPressHardSample8,...,TPressProdReportHardSrel,TPressProdReportDiamVM,TPressProdReportDiamSrel,QProd,TPressSetupObject,TPressSetupCode,TPressSetupValue,EventObject,EventCode,event
0,207304,2022-01-07 11:17:48,155.0,159.0,155.0,165.0,148.0,159.0,168.0,153.0,...,4.19,0.0,0.0,0.0,1580.0,1.0,45.0,1580.0,5002.0,0
1,207304,2022-01-07 11:17:48,155.0,159.0,155.0,165.0,148.0,159.0,168.0,153.0,...,4.19,0.0,0.0,0.0,1580.0,1.0,45.0,1580.0,5101.0,0
3,207304,2022-01-07 11:17:48,155.0,159.0,155.0,165.0,148.0,159.0,168.0,153.0,...,4.19,0.0,0.0,0.0,1580.0,1.0,45.0,1580.0,5108.0,0
4,207304,2022-01-07 11:17:48,155.0,159.0,155.0,165.0,148.0,159.0,168.0,153.0,...,4.19,0.0,0.0,0.0,1580.0,1.0,45.0,1580.0,5118.0,0
5,207304,2022-01-07 11:17:48,155.0,159.0,155.0,165.0,148.0,159.0,168.0,153.0,...,4.19,0.0,0.0,0.0,1580.0,1.0,45.0,1580.0,5106.0,0


# Merged files so far

- TPressReport
- TPressHard
- TPressThick
- TPressWeight
- Events (anomaly detection)

# Files that need to be merged

- Process15...
- Var ??? (see content)
- TPressDiam
- TPressParameter
- TPressSetup